# Terminal-Bench evaluator harness (step 0 toward GRPO)

**Goal of this notebook.** Get the *evaluation pipe* working end-to-end on a single Terminal-Bench task before we touch any RL code. Concretely:

1. Install / import `terminal-bench` and verify the Docker backend.
2. List tasks and inspect one in detail (instruction + grader).
3. Load a small base model (Qwen3-0.6B) and generate a candidate shell-action.
4. Run that candidate through Terminal-Bench's grader to obtain a scalar success reward.
5. Wrap (3)+(4) as a TRL-compatible `reward_fn(completions, **kw) -> List[float]`.

Once this harness returns sensible scalars, plugging it into `GRPOTrainer` is the same wiring as `script/grpo_gsm8k.py` — the only thing that changes is the reward function.

> **References (verify before relying on exact APIs):**
> - <https://github.com/laude-institute/terminal-bench>
> - <https://www.tbench.ai/docs>
> - TRL GRPO docs: <https://huggingface.co/docs/trl/main/en/grpo_trainer>

I've left a few `# TODO(verify)` markers where the upstream API changes
occasionally; check the cell output and adjust the import paths if needed.

## 1. Environment check

Terminal-Bench tasks run inside Docker containers managed by `tmux`, so you need:

- a working **Docker daemon** (`docker info` should succeed);
- the `terminal-bench` package (`pip install terminal-bench` or `uv pip install terminal-bench`);
- a GPU (or CPU + patience) for the model itself. Qwen3-0.6B fits in <2 GB.

The cell below is read-only: it only reports what's missing.

In [1]:
import importlib, shutil, subprocess, sys

def _check(name, kind, probe):
    try:
        ok = probe()
        print(f"  [ok]   {kind:<8s} {name}")
        return ok
    except Exception as e:
        print(f"  [miss] {kind:<8s} {name}  -- {e}")
        return None

_check("docker",          "binary",  lambda: subprocess.check_output(["docker", "--version"]).decode().strip())
_check("docker daemon",   "daemon",  lambda: subprocess.check_output(["docker", "info"], stderr=subprocess.STDOUT, timeout=5).decode().splitlines()[0])
_check("tmux",            "binary",  lambda: subprocess.check_output(["tmux", "-V"]).decode().strip())
_check("terminal_bench",  "py-pkg",  lambda: importlib.import_module("terminal_bench").__version__)
_check("trl",             "py-pkg",  lambda: importlib.import_module("trl").__version__)
_check("transformers",    "py-pkg",  lambda: importlib.import_module("transformers").__version__)
_check("torch",           "py-pkg",  lambda: importlib.import_module("torch").__version__)

import torch
print(f"\n  cuda available: {torch.cuda.is_available()}  device count: {torch.cuda.device_count()}")

  [ok]   binary   docker
  [miss] daemon   docker daemon  -- Command '['docker', 'info']' returned non-zero exit status 1.
  [ok]   binary   tmux
  [miss] py-pkg   terminal_bench  -- No module named 'terminal_bench'


/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


  [ok]   py-pkg   trl
  [ok]   py-pkg   transformers
  [ok]   py-pkg   torch

  cuda available: False  device count: 0


## 2. Terminal-Bench primer

Mental model:

- A **task** = a Dockerised CLI environment + a natural-language instruction + a grader (usually a `tests/` folder run inside the container).
- An **agent** receives the instruction, talks to a `tmux` pane, emits shell commands, and reads back the pane output. Episodes can be many turns.
- The **trial** harness sets up the container, runs the agent until it signals "done" (or hits a turn cap), then runs the grader. Success is a binary `{0, 1}`.

For GRPO we need: `instruction → agent rollout → grader → scalar reward`. Today we'll do exactly that for ONE task with a placeholder one-shot agent.

## 3. Discover and inspect a task

The `terminal-bench` package exposes a dataset object and a CLI. The exact import path has shifted between versions; I show two common ones below, then fall back to the CLI.

In [2]:
import importlib, subprocess, json
from pathlib import Path

def _try_imports(paths):
    for p in paths:
        try:
            mod_path, _, attr = p.rpartition(".")
            mod = importlib.import_module(mod_path)
            obj = getattr(mod, attr)
            print(f"  [ok] {p}")
            return obj
        except Exception as e:
            print(f"  [miss] {p}: {e}")
    return None

# TODO(verify): pin to whichever import path your installed version exposes.
Dataset = _try_imports([
    "terminal_bench.dataset.dataset.Dataset",
    "terminal_bench.harness.dataset.Dataset",
    "terminal_bench.Dataset",
])

tasks = []
if Dataset is not None:
    try:
        ds = Dataset()  # default: ships with the package
        tasks = list(ds)
        print(f"  loaded {len(tasks)} tasks via Dataset()")
    except Exception as e:
        print(f"  Dataset() failed: {e}")

# CLI fallback: enumerate task ids without importing internals.
if not tasks:
    try:
        out = subprocess.check_output(["tb", "tasks", "list"], text=True, timeout=15)
        print(out[:2000])
    except Exception as e:
        print(f"  `tb tasks list` failed: {e}")

  [miss] terminal_bench.dataset.dataset.Dataset: No module named 'terminal_bench'
  [miss] terminal_bench.harness.dataset.Dataset: No module named 'terminal_bench'
  [miss] terminal_bench.Dataset: No module named 'terminal_bench'
  `tb tasks list` failed: [Errno 2] No such file or directory: 'tb'


In [3]:
# Pick the first task and dump its public fields.
if tasks:
    t = tasks[0]
    public = {k: v for k, v in vars(t).items() if not k.startswith("_")}
    for k, v in public.items():
        s = str(v)
        print(f"  {k:>20s}: {s[:200]}{'...' if len(s) > 200 else ''}")
else:
    print("  (no tasks loaded -- adjust the import in the previous cell)")

  (no tasks loaded -- adjust the import in the previous cell)


## 4. Load the policy model

We use the smallest Qwen3 here so the notebook runs on a single consumer GPU. For real GRPO runs, swap in 1.7B / 4B.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.eval()
print(f"  loaded {MODEL_NAME}  ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  loaded Qwen/Qwen3-0.6B  (751.6M params)


## 5. One-shot baseline rollout

Simplest possible "agent": prompt the model with the task instruction and ask
it for a single shell command. We do **not** yet execute it — first we check
the model produces something parseable. The goal at this stage is purely to
exercise the I/O contract.

In [5]:
AGENT_PROMPT = """You are operating in a Linux shell. Read the task and respond
with the single bash command that solves it. Wrap the command in triple
backticks so it can be parsed. Do not explain.

TASK:
{instruction}
"""

def render_prompt(task):
    instr = getattr(task, "instruction", None) or getattr(task, "prompt", None) or str(task)
    return AGENT_PROMPT.format(instruction=instr)

import re
def parse_command(text):
    m = re.search(r"```(?:bash|sh)?\s*\n?(.+?)\n?```", text, re.DOTALL)
    return m.group(1).strip() if m else None

if tasks:
    prompt = render_prompt(tasks[0])
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    text = tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print("---- model output ----")
    print(text)
    print("---- parsed command ----")
    print(parse_command(text))
else:
    print("  (no tasks loaded)")

  (no tasks loaded)


## 6. Run the task through Terminal-Bench's grader

We won't write the Docker / tmux orchestration ourselves — `terminal-bench`
already does that. Two integration paths:

**(a) CLI (most stable across versions).** Call `tb run` as a subprocess with
a plug-in agent that emits commands from our model. The harness writes
results to a JSONL run log we then parse.

**(b) Python API.** Import `Trial` / `Harness` directly. Faster iteration
but the API surface moves around — confirm against your installed version.

Both are sketched below; pick one and discard the other when you've
verified what your install supports.


In [6]:
# (a) CLI path -- stable contract, slower per-call.
# Here we just demonstrate invoking `tb run` against the installed `nop` /
# `oracle` agent so we can confirm the harness is wired up. Plugging in our
# Qwen agent is a separate step (writing a small Agent subclass that calls
# `model.generate` for each turn).
import subprocess, json, tempfile, pathlib

# TODO(verify): exact CLI shape -- check `tb run --help` on your install.
try:
    out = subprocess.run(
        ["tb", "run", "--help"],
        capture_output=True, text=True, timeout=15,
    )
    print(out.stdout or out.stderr)
except FileNotFoundError:
    print("  `tb` CLI not on PATH -- pip install terminal-bench then re-run.")

  `tb` CLI not on PATH -- pip install terminal-bench then re-run.


In [7]:
# (b) Python API path -- faster, more brittle.
# Sketch only: confirm the imports below match your installed version.
Trial = _try_imports([
    "terminal_bench.harness.trial.Trial",
    "terminal_bench.trial.Trial",
    "terminal_bench.Trial",
])
Agent = _try_imports([
    "terminal_bench.agents.base.Agent",
    "terminal_bench.agent.Agent",
    "terminal_bench.Agent",
])
print("\n  fill in the next cell once both imports resolve.")

  [miss] terminal_bench.harness.trial.Trial: No module named 'terminal_bench'
  [miss] terminal_bench.trial.Trial: No module named 'terminal_bench'
  [miss] terminal_bench.Trial: No module named 'terminal_bench'
  [miss] terminal_bench.agents.base.Agent: No module named 'terminal_bench'
  [miss] terminal_bench.agent.Agent: No module named 'terminal_bench'
  [miss] terminal_bench.Agent: No module named 'terminal_bench'

  fill in the next cell once both imports resolve.


## 7. The reward function (TRL contract)

The shape that `GRPOTrainer` expects:

```python
def reward_fn(completions, **kwargs) -> list[float]:
    ...
```

Plus any per-prompt fields you stuffed into the dataset (e.g. `task_id`)
arrive in `**kwargs` aligned with `completions`. Below is the *shape* we
want once the harness above is wired up:

In [8]:
def tbench_success_reward(completions, task_id, **kwargs):
    """One reward per (completion, task_id). 1.0 on grader-pass, 0.0 otherwise."""
    rewards = []
    for completion, tid in zip(completions, task_id):
        text = completion if isinstance(completion, str) else completion[0].get("content", "")
        cmd = parse_command(text)
        if cmd is None:
            rewards.append(0.0)
            continue
        # TODO: replace with real harness call once section 6 is live.
        # success = run_one_task(tid, agent_command=cmd)
        success = False
        rewards.append(1.0 if success else 0.0)
    return rewards

# Smoke test the contract with a fake completion + task_id.
demo = tbench_success_reward(
    completions=["```bash\necho hello\n```"],
    task_id=["dummy-0"],
)
print(demo)

[0.0]


## 8. Next steps (not in this notebook)

- Implement a `QwenAgent(Agent)` subclass that wraps `model.generate` for each
  turn, talking to the `tmux` pane the harness exposes.
- Replace the `success = False` placeholder with a real harness call. Cache
  results keyed by `(task_id, command_hash)` to keep iteration cheap.
- Build a HF `Dataset` whose rows carry `prompt` and `task_id` fields, mirroring
  `script/grpo_gsm8k.py`. Pass `tbench_success_reward` to `GRPOTrainer`.
- Group size `G`: tbench tasks are slow (Docker boot, multi-turn). Start with
  `G=2` and a small task subset before scaling up.
- Caveat: tbench rewards are sparse and binary. Once the binary harness works,
  this is exactly the regime where the predictive-velocity reward we're
  developing in the manuscript would help (D2/D3): score each emitted token
  by its contribution to a *gold solution transcript* rather than waiting for
  the trial-end verdict.